In [ ]:
import importlib
import torch
import numpy as np
import sys
import os
import matplotlib.pyplot as plt

model_name = "2D_Fernandes_Phelan"
relative_path = os.path.join('..', '..', 'dptorch')

notebook_dir = os.getcwd()
absolute_path = os.path.abspath(os.path.join(notebook_dir, relative_path))

sys.path.insert(0, absolute_path)

#### what to load
checkpoint_file = 2999


model = importlib.import_module(f"{model_name}.Model")

# RNG
torch.manual_seed(123)


m = model.SpecifiedModel.load(
    path=os.path.abspath(f"data/2D/Iter_{checkpoint_file}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name},
)

m_prev = model.SpecifiedModel.load(
    path=os.path.abspath(f"data/2D/Iter_{checkpoint_file-1}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name},
)

sigma = m.cfg["model"]["params"]["sigma"]
beta = m.cfg["model"]["params"]["beta"]
n_types = m.cfg["model"]["params"]["n_types"]
gp_offset = m.cfg["model"]["params"]["GP_offset"]

In [ ]:
import logging

shock_lst = np.loadtxt((f"data/2D/2D_shock_lst.txt"))

logging.getLogger("DPGPModel").setLevel(30)

pp = importlib.import_module(f"{model_name}.PostProcess")

output_dir = os.path.join(notebook_dir, "data/2D")
os.makedirs(output_dir, exist_ok=True)

previous_cwd = os.getcwd()
try:
    os.chdir(output_dir)
    sim_path = pp.simulate(m, m_prev, m.cfg, model, shock_lst)
    init_run = True
finally:
    os.chdir(previous_cwd)

